In [11]:
# generate inverted vector V' fo [53, 38, 17, 23, 79]

import numpy as np

# 1. Define the input vector V
V = np.array([53, 38, 17, 23, 79])
V = np.array([0.5, 0.5, -0.5, 0.5])
n = len(V)

print(f"Original Vector V: {V.astype(int)}")
print(f"Mean of V: {int(np.mean(V))}\n")

# 2. Construct the Identity matrix (I) and Averaging matrix (A)
I = np.eye(n)
A = np.full((n, n), 1.0 / n)

# 3. Compute the Diffusion Matrix / Inversion Operator: D = -I + 2A
D = -I + 2 * A

# 4. Apply the transformation and cast the final result to integers
V_prime = D.dot(V)
V_prime_int = V_prime.astype(int)

print("--- Result ---")
print(f"Inverted Vector V': {V_prime_int}")

Original Vector V: [0 0 0 0]
Mean of V: 0

--- Result ---
Inverted Vector V': [0 0 1 0]


In [10]:
# generate matrix  based on input mapping 
import numpy as np

# 1. Ask the user for the number of bits
n = int(input("Enter number of bits (n): "))
N_inputs = 2**n
N_total = 2 ** (n + 1)

# 2. Collect the mapping interactively
mapping = {}
print(f"\nPlease enter 0 or 1 for each of the {N_inputs} inputs:")
for i in range(N_inputs):
    binary_str = format(i, f"0{n}b")
    val = int(input(f"Mapping value for {binary_str}: "))
    mapping[binary_str] = val

# 3. Create the row and column header labels
state_labels = []
for i in range(N_inputs):
    binary_str = format(i, f"0{n}b")
    state_labels.append(f"{binary_str}, 0")
    state_labels.append(f"{binary_str}, 1")

# 4. Compute the Oracle Matrix structure
U_f = np.zeros((N_total, N_total), dtype=int)
for idx, label in enumerate(state_labels):
    x_str, y_str = label.split(", ")
    y = int(y_str)
    y_prime = y ^ mapping[x_str]
    dest_idx = state_labels.index(f"{x_str}, {y_prime}")
    U_f[dest_idx, idx] = 1

# 5. Print out the matrix exactly matching the photo layout
print("\n[")
for i in range(N_total):
    # Replaces 0 with an empty string for the clean look
    row_vals = [str(num) if num != 0 else " " for num in U_f[i]]
    
    # Determines the bracket type depending on the row index
    start_bracket = " [" if i == 0 else "  "
    end_bracket = " ]" if i == N_total - 1 else ""
    
    print(f"{start_bracket}  " + "   ".join(row_vals) + f"{end_bracket}")
print("]")

Enter number of bits (n):  2



Please enter 0 or 1 for each of the 4 inputs:


Mapping value for 00:  0
Mapping value for 01:  0
Mapping value for 10:  1
Mapping value for 11:  0



[
 [  1                            
        1                        
            1                    
                1                
                        1        
                    1            
                            1    
                                1 ]
]


In [13]:
# for 1/2 , 1/2 , -1/2, 1/2

import numpy as np

# 1. Define the input vector V
V = np.array([0.5, 0.5, -0.5, 0.5])
n = len(V)

print(f"Original Vector V: {V.astype(float)}")
print(f"Mean of V: {int(np.mean(V))}\n")

# 2. Construct the Identity matrix (I) and Averaging matrix (A)
I = np.eye(n)
A = np.full((n, n), 1.0 / n)

# 3. Compute the Diffusion Matrix / Inversion Operator: D = -I + 2A
D = -I + 2 * A

# 4. Apply the transformation and cast the final result to integers
V_prime = D.dot(V)
V_prime_int = V_prime.astype(float)

print("--- Result ---")
print(f"Inverted Vector V': {V_prime_int}")

Original Vector V: [ 0.5  0.5 -0.5  0.5]
Mean of V: 0

--- Result ---
Inverted Vector V': [0. 0. 1. 0.]


In [15]:
# modified Uf for 11
import numpy as np
import cirq

# Define the oracle matrix Uf
Uf = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0]
])

# Define the matrix -I + 2A
result_matrix = np.array([
    [-1/2, 1/2, 1/2, 1/2],
    [1/2, -1/2, 1/2, 1/2],
    [1/2, 1/2, -1/2, 1/2],
    [1/2, 1/2, 1/2, -1/2]
])

# Define the number of qubits
n = 3  # Assuming 3 qubits for demonstration purposes

# Initialize the circuit
circuit = cirq.Circuit()


# Step 1: Apply H⊗n
circuit.append(cirq.H.on_each(*cirq.LineQubit.range(n-1)))
circuit.append(cirq.X(cirq.LineQubit(n - 1)))
# Step 2: Repeat √2^n times
sqrt_n = int(np.sqrt(2 ** n))+1
for _ in range(sqrt_n):
    # Step 3a: Apply the phase inversion operation: Uf (I ⊗ H)
 
    circuit.append(cirq.H(cirq.LineQubit(n - 1)))
    circuit.append(cirq.MatrixGate(Uf).on(*cirq.LineQubit.range(n)))
   
    
    # Step 3b: Apply the inversion about the mean operation: -I + 2A
    circuit.append(cirq.MatrixGate(result_matrix).on(*cirq.LineQubit.range(n-1)))

# Step 4: Measure the qubits
circuit.append(cirq.measure(*cirq.LineQubit.range(2), key='result'))

# Print the circuit
print("Grover's Algorithm Circuit:")
print(circuit)

# Simulate the circuit
simulator = cirq.Simulator()
result = simulator.run(circuit, repetitions=3)

# Extract measurement results from the result object
measurement_result = result.measurements['result'][0]

# Print the measurement result
print("\nMeasurement Result:")
print(measurement_result)

final_state_vector = simulator.simulate(circuit).final_state_vector
print(final_state_vector)


Grover's Algorithm Circuit:
              ┌               ┐                           ┌               ┐                           ┌               ┐
              │1 0 0 0 0 0 0 0│                           │1 0 0 0 0 0 0 0│                           │1 0 0 0 0 0 0 0│
              │0 1 0 0 0 0 0 0│   ┌                   ┐   │0 1 0 0 0 0 0 0│   ┌                   ┐   │0 1 0 0 0 0 0 0│   ┌                   ┐
              │0 0 1 0 0 0 0 0│   │-0.5  0.5  0.5  0.5│   │0 0 1 0 0 0 0 0│   │-0.5  0.5  0.5  0.5│   │0 0 1 0 0 0 0 0│   │-0.5  0.5  0.5  0.5│
0: ───H───────│0 0 0 1 0 0 0 0│───│ 0.5 -0.5  0.5  0.5│───│0 0 0 1 0 0 0 0│───│ 0.5 -0.5  0.5  0.5│───│0 0 0 1 0 0 0 0│───│ 0.5 -0.5  0.5  0.5│───M('result')───
              │0 0 0 0 1 0 0 0│   │ 0.5  0.5 -0.5  0.5│   │0 0 0 0 1 0 0 0│   │ 0.5  0.5 -0.5  0.5│   │0 0 0 0 1 0 0 0│   │ 0.5  0.5 -0.5  0.5│   │
              │0 0 0 0 0 1 0 0│   │ 0.5  0.5  0.5 -0.5│   │0 0 0 0 0 1 0 0│   │ 0.5  0.5  0.5 -0.5│   │0 0 0 0 0 1 0 0│   │ 0.5  0.5  0